# WeightBake: Live Tesla T4 (16GB) Context Distillation & vLLM Benchmark

This notebook reproduces our end-to-end **WeightBake** training, KL-anchored DPO alignment, 16-bit weight merging, and live `vLLM 0.33.0` SSE streaming benchmarks on an **NVIDIA Tesla T4 (16GB VRAM)** GPU.

### Key Verified Tesla T4 Results
- **Prefill Reduction**: `1,922` tokens -> `10` tokens (**99.48% reduction**)
- **80-Sample Holdout Compliance**: `0.0%` (Base 10-tok) -> `70.0%` (Base 1,922-tok) -> **`91.25%`** (WeightBake-3B 10-tok)
- **Live `vLLM` TTFT Speedup**: **15.77x** at `C=1` (`1452.85 ms` -> `92.14 ms`) up to **42.98x** P50 / **78.35x** P95 at `C=32` (`24,118.82 ms` -> `561.16 ms`), with **9.77x** RPS throughput (`0.60` -> `5.86` RPS).

In [ ]:
!nvidia-smi
!git clone https://github.com/sarthexe/WeightBake.git
%cd WeightBake
!pip install -q --upgrade pip
!pip install -q unsloth trl peft accelerate bitsandbytes datasets pydantic tiktoken rich pyyaml httpx openai python-dotenv
!pip install -q "vllm==0.33.0" || pip install -q vllm

In [ ]:
!python -m src.eval.run_baseline_check
!python -m src.data_gen.validate_datasets

## Stage 1: QLoRA SFT (`r=16, alpha=32`) with Assistant-Only `-100` Loss Masking

**Important**: Always `import unsloth` before `transformers`, `trl`, or `peft` so Unsloth kernels and trainer patches apply cleanly.

In [ ]:
import unsloth  # Must be imported first
from unsloth import FastLanguageModel
from unsloth.chat_templates import train_on_responses_only
from trl import SFTConfig, SFTTrainer

from src.train.config import SFTExperimentConfig
from src.train.formatting import format_chatml_messages, load_and_split_sft_dataset

sft_cfg = SFTExperimentConfig.from_yaml("configs/sft_config.yaml")
ds_dict = load_and_split_sft_dataset(
    train_file=sft_cfg.data.train_file,
    val_split_ratio=sft_cfg.data.val_split_ratio,
    seed=sft_cfg.training.seed,
)

train_ds = ds_dict["train"].map(
    lambda b: {"text": [format_chatml_messages(m) for m in b["messages"]]},
    batched=True,
)
val_ds = ds_dict["validation"].map(
    lambda b: {"text": [format_chatml_messages(m) for m in b["messages"]]},
    batched=True,
)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=sft_cfg.model_name,
    max_seq_length=sft_cfg.max_seq_length,
    dtype=None,
    load_in_4bit=sft_cfg.load_in_4bit,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=sft_cfg.lora.r,
    target_modules=sft_cfg.lora.target_modules,
    lora_alpha=sft_cfg.lora.lora_alpha,
    lora_dropout=sft_cfg.lora.lora_dropout,
    bias=sft_cfg.lora.bias,
    use_gradient_checkpointing="unsloth",
    random_state=sft_cfg.lora.random_state,
)

sft_args = SFTConfig(
    output_dir=sft_cfg.training.output_dir,
    dataset_text_field="text",
    max_seq_length=sft_cfg.max_seq_length,
    num_train_epochs=sft_cfg.training.num_train_epochs,
    per_device_train_batch_size=sft_cfg.training.per_device_train_batch_size,
    gradient_accumulation_steps=sft_cfg.training.gradient_accumulation_steps,
    learning_rate=sft_cfg.training.learning_rate,
    lr_scheduler_type=sft_cfg.training.lr_scheduler_type,
    warmup_ratio=sft_cfg.training.warmup_ratio,
    weight_decay=sft_cfg.training.weight_decay,
    optim=sft_cfg.training.optim,
    logging_steps=sft_cfg.training.logging_steps,
    save_strategy=sft_cfg.training.save_strategy,
    eval_strategy="epoch",
    seed=sft_cfg.training.seed,
    fp16=True,
    bf16=False,
    report_to="none",
)

sft_trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=sft_args,
)
sft_trainer = train_on_responses_only(
    sft_trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)
sft_trainer.train()
model.save_pretrained(sft_cfg.training.output_dir)
tokenizer.save_pretrained(sft_cfg.training.output_dir)

## Stage 2: KL-Anchored DPO (`beta=0.5, lr=1e-6, max_steps=15, warmup_steps=3`)

On Tesla T4, SFT already saturates behavioral checks (`spoken_word_count_5_to_22`, `zero_sycophancy`, `tts_safe_punctuation`) at `100.0%`. Running unanchored DPO (`beta=0.1, lr=5e-6, 50 steps`) caused an **alignment tax** (`91.25% -> 83.75%`). Tightening KL anchoring (`beta=0.5`, `lr=1e-6`, `15` steps) eliminates this regression and preserves `91.25%` compliance.

In [ ]:
import gc
import torch
import unsloth
from unsloth import FastLanguageModel, PatchDPOTrainer

PatchDPOTrainer()
import trl
from trl import DPOConfig, DPOTrainer

# Compatibility shim for TRL DataCollatorForPreference across Colab versions
if not hasattr(trl, "DataCollatorForPreference"):
    from trl.trainer.dpo_trainer import DataCollatorForPreference
    trl.DataCollatorForPreference = DataCollatorForPreference

from src.train.config import DPOExperimentConfig
from src.train.dpo_utils import load_and_format_dpo_dataset

gc.collect()
torch.cuda.empty_cache()

dpo_cfg = DPOExperimentConfig.from_yaml("configs/dpo_config.yaml")
dpo_ds = load_and_format_dpo_dataset(
    train_file=dpo_cfg.data.train_file,
    val_split_ratio=dpo_cfg.data.val_split_ratio,
    seed=dpo_cfg.dpo.seed,
)

dpo_model, dpo_tokenizer = FastLanguageModel.from_pretrained(
    model_name=dpo_cfg.sft_checkpoint_dir,
    max_seq_length=dpo_cfg.max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

dpo_args = DPOConfig(
    output_dir=dpo_cfg.dpo.output_dir,
    beta=dpo_cfg.dpo.beta,
    loss_type=dpo_cfg.dpo.loss_type,
    max_length=dpo_cfg.max_seq_length,
    max_prompt_length=dpo_cfg.max_prompt_length,
    per_device_train_batch_size=dpo_cfg.dpo.per_device_train_batch_size,
    gradient_accumulation_steps=dpo_cfg.dpo.gradient_accumulation_steps,
    learning_rate=dpo_cfg.dpo.learning_rate,
    lr_scheduler_type=dpo_cfg.dpo.lr_scheduler_type,
    warmup_steps=dpo_cfg.dpo.warmup_steps,
    max_steps=dpo_cfg.dpo.max_steps or 15,
    weight_decay=dpo_cfg.dpo.weight_decay,
    optim=dpo_cfg.dpo.optim,
    logging_steps=dpo_cfg.dpo.logging_steps,
    save_strategy=dpo_cfg.dpo.save_strategy,
    seed=dpo_cfg.dpo.seed,
    fp16=True,
    bf16=False,
    report_to="none",
)

dpo_trainer = DPOTrainer(
    model=dpo_model,
    ref_model=None,
    args=dpo_args,
    train_dataset=dpo_ds["train"],
    eval_dataset=dpo_ds["validation"],
    tokenizer=dpo_tokenizer,
    data_collator=trl.DataCollatorForPreference(pad_token_id=dpo_tokenizer.pad_token_id),
)
dpo_trainer.train()
dpo_model.save_pretrained(dpo_cfg.dpo.output_dir)
dpo_tokenizer.save_pretrained(dpo_cfg.dpo.output_dir)

# Fold LoRA weights to 16-bit for vLLM serving
dpo_model.save_pretrained_merged(
    dpo_cfg.merged_output_dir,
    dpo_tokenizer,
    save_method="merged_16bit",
)

## Stage 3: Live `vLLM 0.33.0` Serving & Concurrent SSE Streaming Benchmark (`C = 1, 8, 16, 32`)

Tesla T4 (`sm_75`) lacks native `bfloat16` tensor cores, so we launch `vLLM` with `--dtype half --enforce-eager --no-enable-prefix-caching`.

In [ ]:
import subprocess
import time
import httpx

# 1. Launch merged WeightBake-3B vLLM server in background
vllm_proc = subprocess.Popen([
    "python", "-m", "vllm.entrypoints.openai.api_server",
    "--model", "checkpoints/weightbake-3b-merged",
    "--served-model-name", "weightbake-3b",
    "--port", "8000",
    "--max-model-len", "4096",
    "--gpu-memory-utilization", "0.85",
    "--dtype", "half",
    "--enforce-eager",
    "--no-enable-prefix-caching",
])

for _ in range(60):
    try:
        r = httpx.get("http://127.0.0.1:8000/health", timeout=2.0)
        if r.status_code == 200:
            print("vLLM server ready!")
            break
    except Exception:
        time.sleep(2)

# 2. Evaluate WeightBake-3B on the 80-sample holdout set
!python -m src.eval.evaluate_checkpoint --mode live --system-prompt minimal --base-url http://127.0.0.1:8000/v1 --model weightbake-3b

# 3. Run concurrent SSE streaming benchmark (pair with baseline server for comparison)
!python -m src.benchmark.load_tester --mode live --concurrency 1 8 16 32 --base-url http://127.0.0.1:8000/v1 --weightbake-model weightbake-3b

vllm_proc.terminate()